In [ ]:
import json

import numpy as np
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("device: ", device)


def set_seed(seed: int):
    # random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=False)

### Гиперпараметры

In [ ]:
SEED = 42
set_seed(SEED)

NUM_CLASSES = 2  # фон / знак

IMAGE_SIZE = 512
VAL_FRACTION = 0.2
BATCH_SIZE = 2
EPOCHS = 20
LR = 4e-4

### Обработка датасета

In [ ]:
def load_via_annotations(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = list(data.values()) if isinstance(data, dict) else data
    annotations = {}
    for record in records:
        filename = record.get("filename")
        if not filename:
            continue

        regions = record.get("regions", {})
        if isinstance(regions, dict):
            regions = list(regions.values())

        annotations[filename] = regions

    return annotations

# В ДАТАСЕТЕ МАСКИ ОБОЗНАЧАЮТСЯ ЧЕРЕЗ ФИГУРЫ И ОПОРНЫЕ ТОЧКИ
# РАЗОБРАТЬСЯ КАК В ИТОГЕ ПРЕДСТАВЛЕНЫ МАСКИ -
def rasterize_region(mask, shape_attributes):
    h, w = mask.shape
    shape_name = shape_attributes.get("name")

    if shape_name == "polygon":
        xs = shape_attributes.get("all_points_x", [])
        ys = shape_attributes.get("all_points_y", [])
        if len(xs) >= 3 and len(xs) == len(ys):
            points = np.array(
                [[round(x), round(y)] for x, y in zip(xs, ys)],
                dtype=np.int32
            )
            points[:, 0] = np.clip(points[:, 0], 0, w - 1)
            points[:, 1] = np.clip(points[:, 1], 0, h - 1)
            cv2.fillPoly(mask, [points], 1)

    elif shape_name == "ellipse":
        cx = int(round(shape_attributes["cx"]))
        cy = int(round(shape_attributes["cy"]))
        rx = max(1, int(round(shape_attributes["rx"])))
        ry = max(1, int(round(shape_attributes["ry"])))
        cv2.ellipse(
            mask,
            (cx, cy),
            (rx, ry),
            0, 0, 360, 1, thickness=-1
        )

    return mask